# Per-Series Feature Importance with Heterogeneous Effects

This notebook demonstrates how **per-series feature importance** can reveal heterogeneous effects that are hidden by global importance analysis.

## The Problem

In multi-series forecasting, a feature may be critically important for some series but completely irrelevant for others. Standard global permutation feature importance (PFI) averages across all series, potentially masking these differences.

## The Solution

Using `explain_per_series()` from the xeries library, we can compute feature importance separately for each series, revealing the true heterogeneous effects.

In [ ]:
import numpy as np
import pandas as pd
import lightgbm as lgb
import matplotlib.pyplot as plt

from xeries import (
    ConditionalPermutationImportance,
    plot_importance_bar,
    plot_importance_heatmap,
    plot_importance_per_series,
)

## 1. Data Generation with Heterogeneous Feature Effects

We generate synthetic data where the importance of feature `X2` is **explicitly different** across two groups of series:

### Data Generating Process (DGP)

**Group A** (series 0-4): Both features matter
```
y[t] = α + 0.6 × X1[t] + 0.8 × X2[t] + ε[t]
```

**Group B** (series 5-9): Only X1 matters
```
y[t] = α + 0.6 × X1[t] + 0.0 × X2[t] + ε[t]
```

Where:
- `α`: Series-specific intercept (random)
- `X1, X2`: Random features in [0, 10]
- `ε`: Gaussian noise with σ=0.5

**Ground Truth**: X1 is globally important, X2 is important only for Group A.

In [ ]:
def generate_heterogeneous_data(n_series=10, n_steps=300, random_state=42):
    """
    Generate synthetic data where X2's importance differs between groups.
    
    Group A (first half): y = α + 0.6*X1 + 0.8*X2 + noise
    Group B (second half): y = α + 0.6*X1 + 0.0*X2 + noise
    """
    np.random.seed(random_state)
    
    all_data = []
    group_split = n_series // 2
    
    for i in range(n_series):
        # Features
        X1 = np.random.rand(n_steps) * 10
        X2 = np.random.rand(n_steps) * 10
        noise = np.random.randn(n_steps) * 0.5
        alpha = np.random.rand() * 5  # Series-specific intercept
        
        # Target depends on group
        if i < group_split:
            # Group A: X2 is important
            group = 'A'
            y = alpha + 0.6 * X1 + 0.8 * X2 + noise
        else:
            # Group B: X2 has no effect
            group = 'B'
            y = alpha + 0.6 * X1 + 0.0 * X2 + noise
        
        series_df = pd.DataFrame({
            'series_id': f'series_{i}',
            'group': group,
            'y': y,
            'X1': X1,
            'X2': X2,
        })
        all_data.append(series_df)
    
    data = pd.concat(all_data, ignore_index=True)
    
    # Add series_code for model training
    series_mapping = {s: i for i, s in enumerate(data['series_id'].unique())}
    data['series_code'] = data['series_id'].map(series_mapping)
    
    return data, series_mapping

# Generate data
data, series_mapping = generate_heterogeneous_data(n_series=10, n_steps=300)

print(f"Data shape: {data.shape}")
print(f"\nSeries and their groups:")
for series_id in data['series_id'].unique():
    group = data[data['series_id'] == series_id]['group'].iloc[0]
    print(f"  {series_id}: Group {group}")

print(f"\nGround truth:")
print(f"  Group A: X2 coefficient = 0.8 (important)")
print(f"  Group B: X2 coefficient = 0.0 (not important)")

## 2. Train a Global LightGBM Model

We train a single model on all series. The model will learn the "average" effect of X2 across all series.

In [ ]:
# Prepare features and target
feature_cols = ['X1', 'X2', 'series_code']
X = data[feature_cols].copy()
y = data['y'].values

# Train LightGBM model
model = lgb.LGBMRegressor(
    n_estimators=100,
    max_depth=6,
    random_state=42,
    verbose=-1
)
model.fit(X, y)

print(f"Model R² score: {model.score(X, y):.4f}")

## 3. Global Feature Importance

First, let's compute **global** permutation feature importance. This aggregates across all series and will show an "averaged" importance for X2.

In [ ]:
# Create explainer
explainer = ConditionalPermutationImportance(
    model=model,
    metric='mse',
    n_repeats=10,
    random_state=42
)

# Compute global importance (only for X1 and X2)
global_result = explainer.explain(
    X=X,
    y=y,
    features=['X1', 'X2']
)

print("=== GLOBAL Feature Importance ===")
print("(Aggregated across ALL series)\n")
print(global_result.to_dataframe().to_string(index=False))

In [ ]:
# Visualize global importance
fig, ax = plot_importance_bar(
    global_result,
    title="Global Feature Importance (All Series Combined)"
)
plt.tight_layout()
plt.show()

### Observation

Global importance shows **moderate** importance for X2. This is misleading!
- X2 is **very important** for Group A (coefficient = 0.8)
- X2 is **completely unimportant** for Group B (coefficient = 0.0)

The global view averages these effects, hiding the true heterogeneity.

## 4. Per-Series Feature Importance

Now let's use `explain_per_series()` to compute importance separately for each series. This will reveal the **true heterogeneous effects**.

In [ ]:
# Compute per-series importance
per_series_results = explainer.explain_per_series(
    X=X,
    y=y,
    series_col='series_code',
    features=['X1', 'X2']
)

# Map numeric codes back to series names
reverse_mapping = {v: k for k, v in series_mapping.items()}
per_series_results = {reverse_mapping[k]: v for k, v in per_series_results.items()}

print(f"Computed importance for {len(per_series_results)} series")

In [ ]:
# Display per-series importance
print("=== PER-SERIES Feature Importance ===")
print("\nX2 Importance by Series and Group:\n")

for series_id in sorted(per_series_results.keys()):
    result = per_series_results[series_id]
    group = data[data['series_id'] == series_id]['group'].iloc[0]
    
    # Get X2 importance
    x2_idx = list(result.feature_names).index('X2')
    x2_importance = result.importances[x2_idx]
    
    marker = "** HIGH **" if x2_importance > 0.5 else "   low   "
    print(f"  {series_id} (Group {group}): X2 importance = {x2_importance:8.4f}  {marker}")

## 5. Visualizations

In [ ]:
# Grid of bar charts for all series
fig, axes = plot_importance_per_series(
    per_series_results,
    max_features=2,
    ncols=5,
    figsize=(16, 6),
    title="Per-Series Feature Importance: Revealing Heterogeneous Effects"
)
plt.tight_layout()
plt.show()

In [ ]:
# Heatmap comparison
fig, ax = plot_importance_heatmap(
    per_series_results,
    features=['X1', 'X2'],
    figsize=(12, 4),
    title="Feature Importance Heatmap: X2 Effect Differs by Group"
)
plt.tight_layout()
plt.show()

In [ ]:
# Compare Group A vs Group B average importance
group_a_series = [s for s in per_series_results.keys() if 'series_' in s and int(s.split('_')[1]) < 5]
group_b_series = [s for s in per_series_results.keys() if 'series_' in s and int(s.split('_')[1]) >= 5]

def avg_importance(series_list, feature):
    values = []
    for s in series_list:
        result = per_series_results[s]
        idx = list(result.feature_names).index(feature)
        values.append(result.importances[idx])
    return np.mean(values)

# Create comparison bar chart
fig, ax = plt.subplots(figsize=(10, 5))

x = np.arange(2)
width = 0.35

group_a_x1 = avg_importance(group_a_series, 'X1')
group_a_x2 = avg_importance(group_a_series, 'X2')
group_b_x1 = avg_importance(group_b_series, 'X1')
group_b_x2 = avg_importance(group_b_series, 'X2')

bars1 = ax.bar(x - width/2, [group_a_x1, group_a_x2], width, label='Group A (X2 important)', alpha=0.8)
bars2 = ax.bar(x + width/2, [group_b_x1, group_b_x2], width, label='Group B (X2 not important)', alpha=0.8)

ax.set_xlabel('Feature')
ax.set_ylabel('Average Importance (MSE increase)')
ax.set_title('Average Feature Importance by Group')
ax.set_xticks(x)
ax.set_xticklabels(['X1', 'X2'])
ax.legend()
ax.axhline(y=0, color='black', linestyle='-', linewidth=0.5)

plt.tight_layout()
plt.show()

print(f"\nAverage X2 Importance:")
print(f"  Group A: {group_a_x2:.4f}")
print(f"  Group B: {group_b_x2:.4f}")
print(f"  Ratio (A/B): {group_a_x2/max(group_b_x2, 0.0001):.2f}x")

## 6. Conclusions

### Key Findings

| Feature | Ground Truth | Global PFI | Per-Series PFI |
|---------|--------------|------------|----------------|
| X1 | Important for all | High | High for all |
| X2 | Important for Group A only | **Moderate (misleading!)** | High for A, ~0 for B |

### Why Per-Series Analysis Matters

1. **Reveals Hidden Heterogeneity**: Global importance showed X2 as moderately important, hiding the fact that it's critical for half the series and irrelevant for the other half.

2. **Enables Targeted Actions**: Knowing which features matter for which series allows for:
   - Feature engineering per group
   - Group-specific models
   - Better understanding of the underlying processes

3. **Model Debugging**: If predictions are poor for specific series, per-series importance helps identify which features the model is (incorrectly) relying on.

### When to Use Per-Series Importance

- Multi-series forecasting with potentially different underlying processes
- When global importance shows moderate values (could be averaging extremes)
- When you suspect feature effects vary across groups or series